# 02 · PPG-only heart-rate estimation

**Goal:** Estimate heart rate from wrist PPG, then compare the predictions with ECG-derived reference labels. No supervised model is trained; the **model** here is a transparent signal-processing baseline.

Two approaches are compared: (A) raw wrist BVP + periodogram; (B) 3rd-order Butterworth bandpass (0.65–3.5 Hz) + Welch spectrum. The strongest in-band peak is converted to BPM: `BPM = 60 × frequency_hz`.

**Critical integrity rule:** reference labels enter only the evaluation stage. They are never passed to `estimate_hr_from_ppg()` or used to select a frequency peak.

> Real-data results must be computed locally. Synthetic mode is clearly marked and is not evidence of dataset performance.

### 1. Environment and dataset configuration

**Default: real UCI data.** Set `PPG_DATA_ROOT` to the directory containing `S1/`, `S2/`, etc. Run from the project root (`jupyter lab`). If you cannot access the archive yet, `PPG_DEMO_MODE=1` enables **synthetic engineering-test data only**. Synthetic output must never be reported as dataset findings.

**Security:** loading the official `S1.pkl` deserializes Python pickle, which can run arbitrary code. Inspect the download and use an isolated, trusted environment. The agent cannot choose a data file or filesystem path.

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython import get_ipython
get_ipython().run_line_magic('matplotlib', 'inline')

HERE = Path.cwd().resolve()
PROJECT_ROOT = HERE if (HERE / 'ppg_demo').is_dir() else HERE.parent
assert (PROJECT_ROOT / 'ppg_demo').is_dir(), 'Run Jupyter from the project root or notebooks/.'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_ROOT = Path(os.getenv('PPG_DATA_ROOT', str(PROJECT_ROOT / 'data' / 'PPG_FieldStudy'))).expanduser()
DEMO_MODE = os.getenv('PPG_DEMO_MODE', '0').lower() in ('1', 'true', 'yes')
SUBJECT = os.getenv('PPG_SUBJECT', 'S1')

from ppg_demo.analysis import (load_subject, validate_subject, PPG_HZ, ACC_HZ,
    WIN_SECONDS, STEP_SECONDS, MIN_HZ, MAX_HZ)
from ppg_demo.synthetic import synthetic_record

validate_subject(SUBJECT)
if DEMO_MODE:
    record = synthetic_record(seconds=1100, subject=SUBJECT, bpm=90, movement_from_seconds=600)
    print('⚠ SYNTHETIC SMOKE TEST — NOT PPG-DaLiA MEASUREMENTS')
else:
    dataset_file = DATA_ROOT / SUBJECT / f'{SUBJECT}.pkl'
    if not dataset_file.is_file():
        raise FileNotFoundError(f'Cannot locate {dataset_file}. Extract the official UCI archive and set PPG_DATA_ROOT.')
    record = load_subject(str(dataset_file.resolve()))
    print(f'Official dataset, participant: {SUBJECT} | source: {dataset_file}')
print('BVP shape:', record.bvp.shape, '| ACC shape:', record.acc.shape, '| reference labels:', record.reference.shape)
plt.rcParams.update({'figure.figsize': (11, 4), 'axes.grid': True, 'grid.alpha': .18,
                     'axes.spines.top': False, 'axes.spines.right': False})

## Method
### 2. Explain the pipeline and select a segment

Use an **8-second window with 2-second stride**, aligned to the reference label index. This baseline is intentionally simple and may lock onto harmonics or movement frequencies. A 4096-point FFT interpolates the spectrum; it does not increase the intrinsic 8-second frequency resolution.

In [ ]:
from ppg_demo.analysis import (analyze_loaded_subject, filter_bvp,
    estimate_hr_from_ppg)
from scipy import signal

START_SECONDS = 30
DURATION_SECONDS = 120
result = analyze_loaded_subject(record, START_SECONDS, DURATION_SECONDS, return_trace=True)
windows = pd.DataFrame(result['windows'])
assert result['complete_windows'] == len(windows)
print('Mode:', 'SYNTHETIC DEMO' if DEMO_MODE else 'OFFICIAL UCI DATA')
print('Number of complete 8s windows:', len(windows))
print('Input signals:', result['estimator_input'])
print('Reference policy:', result['reference_usage'])
display(windows.head(8))

## Signal processing
### 3. Compare raw and filtered PPG for one short excerpt

Filtering is applied to the complete recording before slicing each window, avoiding artificial filtering edges in individual 8-second windows.

In [ ]:
filtered_bvp = result['_plot_signals']['filtered']
first_start = int(windows.iloc[0]['start_seconds'] * PPG_HZ)
span = min(16 * PPG_HZ, len(record.bvp) - first_start)
t = (first_start + np.arange(span)) / PPG_HZ
fig, ax = plt.subplots()
ax.plot(t, record.bvp[first_start:first_start+span], lw=.7, alpha=.65, label='Raw BVP')
ax.plot(t, filtered_bvp[first_start:first_start+span], lw=1.0, label='Filtered BVP')
ax.set(title=f'{SUBJECT}: raw vs filtered BVP — {"SYNTHETIC" if DEMO_MODE else "dataset"}',
       xlabel='Time (s)', ylabel='BVP (recorded device units)')
ax.legend()
plt.show()

### 4. Inspect the first 8-second spectral peak

A strong peak is not automatically a true pulse. In active periods, an accelerometer peak near the PPG peak is a warning sign, not a confirmed artifact.

In [ ]:
eight_seconds = filtered_bvp[first_start:first_start + WIN_SECONDS*PPG_HZ]
freq, power = signal.welch(eight_seconds, fs=PPG_HZ,
                           nperseg=eight_seconds.size, nfft=4096)
mask = (freq >= MIN_HZ) & (freq <= MAX_HZ)
predicted_bpm = windows.iloc[0]['filtered_bpm']
fig, ax = plt.subplots()
ax.plot(60*freq[mask], power[mask])
if pd.notna(predicted_bpm):
    ax.axvline(predicted_bpm, linestyle='--', label=f'Peak: {predicted_bpm:.1f} BPM')
ax.set(title=f'First window: filtered PPG power spectrum — {"SYNTHETIC" if DEMO_MODE else "dataset"}',
       xlabel='Candidate pulse frequency (BPM)', ylabel='Power spectral density')
ax.legend()
plt.show()

## Compare predictions against reference (evaluation only)
### 5. Plot the three time series

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
for col, label, linestyle in [
    ('reference_bpm', 'ECG-derived reference (evaluation only)', '-'),
    ('raw_bpm', 'Raw BVP baseline', '--'),
    ('filtered_bpm', 'Filtered BVP + Welch', '-.'),
]:
    ax.plot(windows['midpoint_seconds'], windows[col],
            linestyle=linestyle, marker='.', markersize=3, label=label)
ax.set(title=f'{SUBJECT}: heart-rate baseline versus reference — {"SYNTHETIC" if DEMO_MODE else "dataset"}',
       xlabel='Window midpoint (s)', ylabel='Heart rate (BPM)')
ax.legend()
plt.show()

### 6. Calculate MAE and signed error

`MAE = mean(abs(predicted BPM - reference BPM))`. MAE is calculated **only** for windows with valid estimates and valid reference labels. Signed error reveals consistent over- or under-estimation.

In [ ]:
comparison = pd.DataFrame([
    {'approach': 'Raw BVP + periodogram', **result['raw_periodogram']},
    {'approach': 'Bandpass + Welch', **result['filtered_welch']},
])
display(comparison)
print('Warning: these metrics are from a single selected participant/segment;')
print('overlapping windows are correlated, and this is not a subject-independent benchmark.')

## Takeaways to write after real execution

- Report the actual MAE and bias of both approaches, including when filtering **does not help**.
- Describe any unstable periods in the plotted predictions and whether wrist movement coincides.
- Limitations: movement-related frequency overlap, higher harmonics, brief windows, parameter choices and uncalibrated quality heuristics.

**Next →** `03_evaluation_error_analysis.ipynb` compares multiple segments and failure cases.